# 第11回　モデル選択とAIC
## ―― 当てはまりの良さ vs 複雑さ、オッカムの剃刀

統計学Ⅰ（B）　／　北星学園大学

第10回で「変数を足せばR²は上がる」と分かった。じゃあ全部入れればいい？　注目は ――

> 良いモデルとは「**当てはまり**」が良いのではなく、「**まだ見ぬデータをよく予測する**」モデルだ。

### フック

> 変数を増やせば R² は上がり続ける。
> **じゃあ思いつく変数を全部入れればいい？　それで“未来”は当たるの？**

In [ ]:
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401
import statsmodels.api as sm

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)
except Exception:
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan
df = df.dropna().reset_index(drop=True)
y = df["テスト点"]
# テスト点と無関係な「でたらめ列」を20本用意（過学習を起こす材料）
rng = np.random.default_rng(1)
でたらめ = pd.DataFrame({f"でたらめ{j}": rng.normal(0,1,len(df)) for j in range(20)})
本物 = df[["勉強時間h","睡眠時間h","SNS時間h"]]
print("準備OK")

---
## 1. 過学習を、目で見る

データを **訓練用（7割）** と **テスト用（3割）** に分ける。訓練用だけでモデルを作り、**見せていないテスト用**でどれだけ予測できるかを測る。

本物の変数に「でたらめ列」を足していきながら、
- **訓練R²**（手元への当てはまり）
- **テスト誤差 RMSE**（未知データの予測のズレ。小さいほど良い）

の両方を追う。

In [ ]:
idx = rng.permutation(len(df)); cut = int(len(df)*0.7)
tr, te = idx[:cut], idx[cut:]

ks = [0, 2, 5, 10, 15, 20]
訓練R2, テストRMSE = [], []
for k in ks:
    X = pd.concat([本物, でたらめ.iloc[:, :k]], axis=1)
    m = sm.OLS(y.iloc[tr], sm.add_constant(X.iloc[tr])).fit()
    Xte = sm.add_constant(X.iloc[te], has_constant="add")
    rmse = np.sqrt(np.mean((y.iloc[te] - m.predict(Xte))**2))
    訓練R2.append(m.rsquared); テストRMSE.append(rmse)
    print(f"でたらめ {k:2d}本: 訓練R²={m.rsquared:.3f}  テストRMSE={rmse:.2f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(ks, 訓練R2, "o-", color="#00897b"); ax[0].set_title("訓練R²（手元への当てはまり）")
ax[0].set_xlabel("でたらめ列の数"); ax[0].set_ylabel("R²")
ax[1].plot(ks, テストRMSE, "o-", color="#e8503a"); ax[1].set_title("テスト誤差RMSE（未知データの予測）")
ax[1].set_xlabel("でたらめ列の数"); ax[1].set_ylabel("RMSE（小さいほど良い）")
plt.suptitle("当てはまりは上がり続けるのに、予測は途中から悪化＝過学習"); plt.tight_layout(); plt.show()

**訓練R²はどんどん上がる**（手元に合わせるのが上手くなる）のに、**テスト誤差は途中から悪化**する。これが **過学習**：手元のデータに合わせすぎて、未知のデータに弱くなる。

つまり「当てはまりの良さ（R²）」と「予測の良さ」は別物。R²を最大化してはいけない。

---
## 2. AIC ―― テストデータ無しで「予測の良さ」を見積もる

毎回データを分けるのは大変。**AIC（赤池情報量規準）** は、**手元のデータだけ**から「予測の良さ」を見積もる指標だ。

$$ \text{AIC} = \underbrace{-2 \times (\text{当てはまりの良さ})}_{\text{小さいほど当てはまり良}} + \underbrace{2 \times (\text{パラメータの数})}_{\text{複雑さのペナルティ}} $$

- 当てはまりが良いほど第1項が小さくなる（AIC下がる）
- 変数を増やすほど第2項が大きくなる（AIC上がる＝ペナルティ）
- **AICが小さいモデルほど良い**。＝「当てはまり」と「複雑さ」のバランスが最良。

これは **オッカムの剃刀**（同じ説明力なら単純なほうを選べ）の定量化だ。

In [ ]:
# 全データで3つのモデルのAICを比べる（AICはテスト分割が要らない）
モデル = {
    "M1 勉強のみ":          ["勉強時間h"],
    "M2 勉強+睡眠+SNS":     ["勉強時間h","睡眠時間h","SNS時間h"],
    "M3 M2+でたらめ10本":   ["勉強時間h","睡眠時間h","SNS時間h"],
}
全変数 = pd.concat([本物, でたらめ], axis=1)
print(f"{'モデル':<22}{'R²(当てはまり)':>14}{'AIC(小さいほど良い)':>20}")
for name, cols in モデル.items():
    use = cols + ([f"でたらめ{j}" for j in range(10)] if "でたらめ" in name else [])
    m = sm.OLS(y, sm.add_constant(全変数[use])).fit()
    print(f"{name:<22}{m.rsquared:>12.3f}{m.aic:>18.1f}")

結果（数値は環境で多少変わるが傾向は同じ）：

| モデル | R² | AIC |
|---|---|---|
| M1 勉強のみ | 0.363 | 2896.4 |
| M2 勉強+睡眠+SNS | 0.377 | **2891.2（最小）** |
| M3 M2+でたらめ10本 | **0.386（最大）** | 2905.8 |

**R²が最大なのは M3（でたらめ入り）。でもAICが最小＝採用すべきは M2。** 両者は一致しない！

- M1→M2：本物の変数（睡眠・SNS）を足したらAICは下がった（＝役に立つ変数は歓迎）。
- M2→M3：でたらめを足したらR²は上がったがAICは上がった（＝中身のない複雑さは罰する）。

AICは「R²の罠」に引っかからず、過学習しないモデルを選んでくれる。

---
## 3. AICの使い方の作法

- **AICは相対指標**。**絶対値そのものに意味はない**（2891という数字単体は何も語らない）。**モデル間の差**で比べる。
- 差が大きいほど優劣は明確（目安：差が2以上で意味あり、10以上で決定的）。
- 比べるモデルは**同じデータ・同じ目的変数**で作ること。
- 「AICが小さい」＝「予測が良い」の見積もりであって、**真のモデルを当てる保証ではない**。あくまで候補の中での相対的なベスト。

> ❌ よくある誤り：「AICの絶対値が小さい/大きいことに意味がある」「変数は多いほど良い」。

---
## 今日のまとめ

| 概念 | ひとこと |
|---|---|
| 過学習 | 手元に合わせすぎて未知データに弱くなる（訓練R²↑なのにテスト誤差↑） |
| 当てはまり vs 予測 | R²最大化はダメ。狙うのは未知データの予測 |
| AIC | −2×当てはまり ＋ 2×パラメータ数。**小さいほど良い** |
| オッカムの剃刀 | 同じ説明力なら単純なモデルを選ぶ。AICはその定量化 |
| ❌ 誤り | AICの絶対値に意味がある／変数は多いほど良い |

> **R²最大のモデルとAIC最小のモデルは違う。**
> 良いモデルは「当てはまり」ではなく「当てはまりと複雑さのバランス」で選ぶ。

**課題（Moodle）**：複数モデルのAICを比較して、採用すべきモデルと理由を述べる。